# Hyperparameter Sensitivity (Figure 4)

Reproduces **Figure 4** from the DTR paper.

This analysis examines the sensitivity of DTR's correlation with accuracy to its two
hyperparameters:
- **g** (settling threshold): fraction of layers that must have JSD below rho to consider a token "settled"
- **rho** (JSD threshold): the JSD cutoff for considering a layer "settled"

The paper shows that the default values (g=0.5, rho=0.85) are robust across benchmarks.

In [ ]:
%matplotlib inline

import sys
from pathlib import Path

sys.path.insert(0, str(Path("..") / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

from dtr.metrics.dtr import compute_dtr
from dtr.analysis.sensitivity import sweep_dtr_params, recompute_dtr_from_jsd
from dtr.analysis.correlation import compute_binned_correlation
from dtr.utils.io import load_json

## Configuration

In [ ]:
BENCHMARKS = ["aime_2025", "hmmt_2025", "gpqa_diamond", "livecodebench"]
BENCHMARK_LABELS = ["AIME 2025", "HMMT 2025", "GPQA Diamond", "LiveCodeBench"]

MODEL = "deepseek-r1"
METRICS_DIR = Path("..") / "results" / "metrics"
SENSITIVITY_PATH = METRICS_DIR / "sensitivity_sweep.json"

# Sweep ranges
G_VALUES = np.arange(0.1, 1.0, 0.1)
RHO_VALUES = [0.70, 0.75, 0.80, 0.85, 0.90, 0.95]

# Default values from the paper
DEFAULT_G = 0.5
DEFAULT_RHO = 0.85

## Load or Compute Sensitivity Sweep

If pre-computed results exist, load them. Otherwise, recompute from JSD matrices.

In [ ]:
if SENSITIVITY_PATH.exists():
    sweep_results = load_json(str(SENSITIVITY_PATH))
    print(f"Loaded pre-computed sensitivity sweep from {SENSITIVITY_PATH}")
else:
    print("Pre-computed sweep not found. Computing from JSD matrices...")
    sweep_results = {}
    
    for bench in BENCHMARKS:
        metrics_path = METRICS_DIR / MODEL / bench / "per_sample_metrics.json"
        if not metrics_path.exists():
            print(f"  Skipping {bench}: metrics not found")
            continue
        
        data = load_json(str(metrics_path))
        accuracies = np.array([d["correct"] for d in data], dtype=float)
        
        bench_results = sweep_dtr_params(
            data,
            accuracies,
            g_values=G_VALUES.tolist(),
            rho_values=RHO_VALUES,
        )
        sweep_results[bench] = bench_results
        print(f"  Computed sweep for {bench}")

print(f"\nSweep results available for: {list(sweep_results.keys())}")

## Figure 4: DTR Correlation vs g for Different rho Values

Each subplot corresponds to a benchmark. Lines show how correlation changes with g
for different values of rho. The default (g=0.5, rho=0.85) is marked.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4.5), sharey=True)
fig.suptitle(
    "Figure 4: DTR Hyperparameter Sensitivity",
    fontsize=16, fontweight="bold", y=1.05,
)

palette = sns.color_palette("viridis", len(RHO_VALUES))
markers = ["o", "s", "D", "^", "v", "P"]

for ax, bench, bench_label in zip(axes, BENCHMARKS, BENCHMARK_LABELS):
    if bench not in sweep_results:
        ax.text(0.5, 0.5, "No data", ha="center", va="center", transform=ax.transAxes)
        ax.set_title(bench_label, fontsize=12)
        continue
    
    results = sweep_results[bench]
    
    for i, rho in enumerate(RHO_VALUES):
        rho_key = f"{rho:.2f}" if isinstance(list(results.keys())[0], str) else rho
        if rho_key not in results and str(rho) not in results:
            continue
        
        rho_data = results.get(rho_key, results.get(str(rho), {}))
        g_vals = sorted([float(g) for g in rho_data.keys()])
        corrs = [rho_data[str(g) if isinstance(list(rho_data.keys())[0], str) else g] for g in g_vals]
        
        ax.plot(
            g_vals, corrs,
            marker=markers[i % len(markers)],
            color=palette[i],
            linewidth=1.5,
            markersize=5,
            label=f"rho={rho:.2f}",
        )
    
    # Mark default
    ax.axvline(DEFAULT_G, color="red", linestyle=":", alpha=0.5, label=f"g={DEFAULT_G} (default)")
    
    ax.set_title(bench_label, fontsize=12)
    ax.set_xlabel("g (settling threshold)", fontsize=11)
    if ax == axes[0]:
        ax.set_ylabel("Pearson r (DTR vs Accuracy)", fontsize=11)
    ax.set_xlim(0.05, 0.95)

# Single legend for all subplots
handles, labels = axes[-1].get_legend_handles_labels()
fig.legend(
    handles, labels,
    loc="lower center",
    ncol=len(RHO_VALUES) + 1,
    fontsize=9,
    bbox_to_anchor=(0.5, -0.1),
)

plt.tight_layout()
plt.savefig("../results/figures/fig4_hyperparam_sensitivity.pdf", bbox_inches="tight", dpi=300)
plt.show()

## 2D Heatmap View

Alternative visualization: a heatmap of correlation strength over the full (g, rho) grid
for each benchmark, making it easy to see the optimal region.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharey=True)
fig.suptitle(
    "DTR Correlation Heatmap: g vs rho",
    fontsize=14, fontweight="bold", y=1.02,
)

for ax, bench, bench_label in zip(axes, BENCHMARKS, BENCHMARK_LABELS):
    if bench not in sweep_results:
        ax.text(0.5, 0.5, "No data", ha="center", va="center", transform=ax.transAxes)
        ax.set_title(bench_label, fontsize=11)
        continue
    
    results = sweep_results[bench]
    
    # Build 2D matrix
    heat_matrix = np.full((len(RHO_VALUES), len(G_VALUES)), np.nan)
    for i, rho in enumerate(RHO_VALUES):
        rho_key = f"{rho:.2f}" if isinstance(list(results.keys())[0], str) else rho
        rho_data = results.get(rho_key, results.get(str(rho), {}))
        for j, g in enumerate(G_VALUES):
            g_key = f"{g:.1f}" if isinstance(list(rho_data.keys())[0], str) else g
            if g_key in rho_data or str(g) in rho_data:
                heat_matrix[i, j] = rho_data.get(g_key, rho_data.get(str(g)))
    
    sns.heatmap(
        heat_matrix, ax=ax,
        xticklabels=[f"{g:.1f}" for g in G_VALUES],
        yticklabels=[f"{rho:.2f}" for rho in RHO_VALUES],
        cmap="YlOrRd", annot=True, fmt=".2f", annot_kws={"size": 7},
        cbar=ax == axes[-1],
        cbar_kws={"label": "Pearson r"} if ax == axes[-1] else {},
    )
    
    ax.set_title(bench_label, fontsize=11)
    ax.set_xlabel("g" if ax == axes[1] or ax == axes[2] else "")
    if ax == axes[0]:
        ax.set_ylabel("rho")

plt.tight_layout()
plt.show()

## Key Takeaway

The DTR metric is robust to hyperparameter choices. The default values (g=0.5, rho=0.85)
consistently achieve near-optimal correlation across all benchmarks, confirming that DTR
does not require per-benchmark tuning.